<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook checks how Phase 1 long-read participants with race recorded as PMI: Skip sit in genetic ancestry space. It queries Cohort Builder person records, overlays those participants on the official v7 ancestry PCA, and flags samples with unusual PC1 values.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import hail as hl
from hail.plot import show

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits import mplot3d

In [ ]:
import os
import pandas as pd

In [ ]:
# Start Hail, using GRCh38 coordinates.
hl.init(default_reference='GRCh38')

In [ ]:
# Controlled Tier short-read ancestry outputs: PCA loadings, and per-participant predicted ancestry and PC scores.
loadings = hl.read_table('gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/loadings.ht')

In [ ]:
ancestry_preds_tbl = hl.import_table(
    'gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv',
    types={ 'pca_features': hl.tarray(hl.tfloat64) }
)

ancestry_preds_tbl.show(20)

In [ ]:
# Predicted ancestry label for every short-read participant, and the count per label.
l1 = ancestry_preds_tbl.ancestry_pred_other.collect()

In [ ]:
Counter(l1)

In [ ]:
# Dataset Builder queries: demographics and survey answers for the long-read participants.
# This query represents dataset "Long reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
            )"""

dataset_86956725_person_df = pd.read_gbq(
    dataset_86956725_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                SELECT
                    DISTINCT concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (
                        select
                            cast(cr.id as string) as id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                        WHERE
                            concept_id IN (
                                1586134
                            ) 
                            AND domain_id = 'SURVEY'
                    ) a 
                        ON (
                            c.path like CONCAT('%',
                        a.id,
                        '.%')) 
                    WHERE
                        domain_id = 'SURVEY' 
                        AND type = 'PPI' 
                        AND subtype = 'QUESTION'
                    )
            )  
            AND (
                answer.PERSON_ID IN (
                    SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_whole_genome_variant = 1 
                        ) 
                        AND cb_search_person.person_id IN (
                            SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                            WHERE
                                has_lr_whole_genome_variant = 1 
                        ) 
                    )
                )"""

# dataset_86956725_survey_df = pd.read_gbq(
#     dataset_86956725_survey_sql,
#     dialect="standard",
#     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
#     progress_bar_type="tqdm_notebook")

# This query represents dataset "Long reads" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_86956725_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string,
        0,
        STRPOS(observation.value_as_string,
        '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                ) 
            ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_86956725_zip_code_socioeconomic_df = pd.read_gbq(
    dataset_86956725_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
# Long-read participants whose self-reported race is 'PMI: Skip' (skipped the question).
pmi_skip_ids = list(dataset_86956725_person_df[dataset_86956725_person_df['race'] == 'PMI: Skip']['person_id'].astype(str))

In [ ]:
# Predicted ancestry for those participants.
ancestry_preds_skip_tbl = ancestry_preds_tbl.filter(hl.literal(pmi_skip_ids).contains(ancestry_preds_tbl.research_id))
l2 = ancestry_preds_skip_tbl.ancestry_pred_other.collect()

In [ ]:
ancestry_preds_df = ancestry_preds_tbl.to_pandas()

In [ ]:
ancestry_preds_df[ancestry_preds_df['research_id'].astype(str).isin(pmi_skip_ids)]

In [ ]:
# First three PC scores for all participants and for the 'PMI: Skip' long-read participants.
pc1 = ancestry_preds_tbl.pca_features[0].collect()
pc2 = ancestry_preds_tbl.pca_features[1].collect()
pc3 = ancestry_preds_tbl.pca_features[2].collect()

In [ ]:
skip_pc1 = ancestry_preds_skip_tbl.pca_features[0].collect()
skip_pc2 = ancestry_preds_skip_tbl.pca_features[1].collect()
skip_pc3 = ancestry_preds_skip_tbl.pca_features[2].collect()

In [ ]:
rids = ancestry_preds_tbl.research_id.collect()

In [ ]:
# Long-read participants with PC1 > 0.05 (outliers on the PCA plot), then their ancestry predictions and
# demographics.
person_ids = list(dataset_86956725_person_df['person_id'].astype(str))

outliers = []
for i in range(len(rids)):
    if rids[i] in person_ids and pc1[i] > 0.05:
        outliers.append(str(rids[i]))
        
outliers

In [ ]:
ancestry_preds_df[ancestry_preds_df['research_id'].astype(str).isin(outliers)]

In [ ]:
dataset_86956725_person_df[dataset_86956725_person_df['person_id'].astype(str).isin(outliers)].sort_values(by=['person_id'])


In [ ]:
# Colours for each continental ancestry group, and for the highlighted participants.
color_map = {
    'afr': '#D573BE',
    'amr': '#519D3E',
    'eas': '#C53A32',
    'eur': '#3976AF',
    'mid': '#EF8536',
    'sas': '#866BE8',
    'oth': '#84584E'
}

outer_color = "#333333"

#pb_color = "#EFC94C"
pb_color = "#FFFFFF"

In [ ]:
# PC1 vs PC2 of all participants coloured by predicted ancestry, with the 'PMI: Skip' long-read
# participants overlaid.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    plt.scatter(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        color=color,
        alpha=0.1,
        s=2,
        label=pop  # Label for legend
    )

a1 = plt.scatter([skip_pc1[0]], [skip_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio')
    
plt.scatter(
    skip_pc1,
    skip_pc2,
    marker="D",
    edgecolor=outer_color,
    color=pb_color,
    s=20
)

plt.xlabel("PC1")
plt.ylabel("PC2")

legend1_patches = [mpatches.Patch(color=color, label=pop.upper()) for pop, color in color_map.items()]
legend1 = plt.legend(handles=legend1_patches)
plt.gca().add_artist(legend1)

legend2 = plt.legend(handles=[a1], loc='lower left')
plt.gca().add_artist(legend2)

plt.show()